In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/project/Bengali_Banglish_80K_Dataset.xlsx


In [2]:
import torch
import gc
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from collections import defaultdict
from tqdm import tqdm

# Load dataset
file_path = '/kaggle/input/project/Bengali_Banglish_80K_Dataset.xlsx'
df = pd.read_excel(file_path)
df = df[[df.columns[0], df.columns[1]]]
df.columns = ['label', 'text']

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
num_classes = df['label'].nunique()
target_list = le.classes_
print(target_list)

# Train/val/test split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Constants
MAX_LEN = 128
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32
TEST_BATCH_SIZE = 32
EPOCHS = 12
LEARNING_RATE = 1e-5

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        text = " ".join(text.split())  # Remove extra whitespaces
        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=True,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'token_type_ids': inputs["token_type_ids"].flatten(),
            'targets': torch.tensor(self.labels[index], dtype=torch.long),  # Single class label
            'text': text
        }

def create_loaders(tokenizer):
    train_dataset = CustomDataset(train_df, tokenizer, MAX_LEN)
    val_dataset = CustomDataset(val_df, tokenizer, MAX_LEN)
    test_dataset = CustomDataset(test_df, tokenizer, MAX_LEN)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
    val_loader =torch.utils.data.DataLoader(val_dataset, batch_size=VALID_BATCH_SIZE, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, test_loader

## Device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

def loss_fn(outputs, targets):
    return nn.CrossEntropyLoss()(outputs, targets)

# Model class for BanglaBERT
class BanglaBERTClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super(BanglaBERTClassifier, self).__init__()
        self.bert_model = AutoModel.from_pretrained(model_name, return_dict=True)
        self.dropout = torch.nn.Dropout(0.3)
        self.linear = torch.nn.Linear(768, num_classes)

    def forward(self, input_ids, attn_mask, token_type_ids):
        output = self.bert_model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            token_type_ids=token_type_ids
        )
        cls_representation = output.last_hidden_state[:, 0, :]  # [CLS] token
        output_dropout = self.dropout(cls_representation)
        output = self.linear(output_dropout)
        return output

# Model class for XLM-Roberta
class XLMRClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super(XLMRClassifier, self).__init__()
        self.bert_model = AutoModel.from_pretrained(model_name, return_dict=True)
        self.dropout = torch.nn.Dropout(0.3)
        self.linear = torch.nn.Linear(768, num_classes)

    def forward(self, input_ids, attn_mask, token_type_ids=None):
        output = self.bert_model(
            input_ids=input_ids,
            attention_mask=attn_mask
        )
        cls_representation = output.last_hidden_state[:, 0, :]  # [CLS] token
        output_dropout = self.dropout(cls_representation)
        output = self.linear(output_dropout)
        return output


def train_model(train_loader, model, optimizer, accumulation_steps=1):
    model.train()
    total_loss = 0
    total_correct = 0
    total = 0
    loop = tqdm(train_loader, desc="Training", leave=False)
    for batch_idx, batch in enumerate(loop):
        ids = batch['input_ids'].to(device, dtype=torch.long, non_blocking=True)
        mask = batch['attention_mask'].to(device, dtype=torch.long, non_blocking=True)
        token_type_ids = batch['token_type_ids'].to(device, dtype=torch.long, non_blocking=True)
        targets = batch['targets'].to(device, dtype=torch.long, non_blocking=True)

        outputs = model(ids, mask, token_type_ids)
        loss = loss_fn(outputs, targets) / accumulation_steps
        total_loss += loss.item()

        _, preds = torch.max(outputs, dim=1)
        total_correct += (preds == targets).sum().item()
        total += targets.size(0)
        loss.backward()
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        loop.set_postfix(loss=loss.item())

    return model, total_correct / total, total_loss / len(train_loader)


from collections import Counter, defaultdict

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, roc_auc_score
)
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import torch.nn.functional as F

def eval_model(loader, model, target_list=None):
    model.eval()
    final_preds = []
    final_targets = []
    probs_all = []
    losses = []

    loop = tqdm(loader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for batch in loop:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            targets = batch['targets'].to(device)

            outputs = model(ids, mask, token_type_ids)
            loss = loss_fn(outputs, targets)
            losses.append(loss.item())

            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            final_preds.extend(preds.cpu().numpy())
            final_targets.extend(targets.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

            loop.set_postfix(loss=loss.item())

    # Metrics
    acc = accuracy_score(final_targets, final_preds)
    f1 = f1_score(final_targets, final_preds, average='weighted')
    precision = precision_score(final_targets, final_preds, average='weighted')
    recall = recall_score(final_targets, final_preds, average='weighted')
    average_loss = np.mean(losses)

    try:
        auc = roc_auc_score(
            y_true=np.eye(len(set(final_targets)))[final_targets],
            y_score=probs_all,
            average='weighted',
            multi_class='ovr'
        )
    except ValueError:
        auc = float('nan')  # Handle edge cases if only one class present

    print("\n=== Final Test Metrics ===")
    print(f"Accuracy       : {acc:.4f}")
    print(f"F1 Score       : {f1:.4f}")
    print(f"Precision      : {precision:.4f}")
    print(f"Recall         : {recall:.4f}")
    print(f"AUC ROC Score  : {auc:.4f}")
    print(f"Average Loss   : {average_loss:.4f}")

    print("\n=== Classification Report ===")
    print(classification_report(
        final_targets, final_preds, target_names=target_list if target_list is not None else None))

    print("\n=== Per-Class Metrics ===")
    final_targets_bin = label_binarize(final_targets, classes=range(num_classes))

    for cls_id in range(num_classes):
        cls_name = target_list[cls_id] if target_list is not None else str(cls_id)
        cls_targets = np.array(final_targets) == cls_id
        cls_preds = np.array(final_preds) == cls_id
        cls_acc = (cls_preds == cls_targets).sum() / len(cls_targets)
        cls_auc = roc_auc_score(final_targets_bin[:, cls_id], np.array(probs_all)[:, cls_id])
        print(f"Class {cls_id} ({cls_name}): Accuracy = {cls_acc:.4f}, AUC = {cls_auc:.4f}")

    return {
        'accuracy': round(acc, 4),
        'f1': round(f1, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'auc': round(auc, 4),
        'loss': round(average_loss, 4)
    }


# ---------------- BanglaBERT ----------------
print(f"\nTraining: BanglaBERT\n" + "=" * 50)
banglabert_tokenizer = AutoTokenizer.from_pretrained("sagorsarker/bangla-bert-base")
train_loader, val_loader, test_loader = create_loaders(banglabert_tokenizer)

banglabert_model = BanglaBERTClassifier("sagorsarker/bangla-bert-base", num_classes).to(device)
optimizer = optim.AdamW(banglabert_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
start = time.time()
best_f1 = 0.0
history = defaultdict(list)

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS} [BanglaBERT]")
    banglabert_model, train_acc, train_loss = train_model(train_loader, banglabert_model, optimizer)
    val_metrics = eval_model(val_loader, banglabert_model, target_list)
    val_f1 = val_metrics['f1']
    val_loss = val_metrics['loss']

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_f1'].append(val_f1)
    history['val_loss'].append(val_loss)

    scheduler.step()

    if val_f1 > best_f1:
        torch.save(banglabert_model.state_dict(), "banglabert_model.bin")
        best_f1 = val_f1
        print("✅ Best BanglaBERT model saved!")
end = time.time()
print(f"⏱️ Total BanglaBERT training & evaluation time: {end - start:.2f} seconds")

print("\nTesting Best BanglaBERT Model...")
banglabert_model.load_state_dict(torch.load("banglabert_model.bin"))

start = time.time()  # Start test-time
metrics=eval_model(test_loader, banglabert_model,target_list)
print(metrics)
end = time.time()
print(f"🧪 BanglaBERT test-set evaluation time: {end - start:.2f} seconds")

del banglabert_model
torch.cuda.empty_cache()
gc.collect()


print(f"\nTraining: XLM-Roberta\n" + "=" * 50)
xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
train_loader, val_loader, test_loader = create_loaders(xlmr_tokenizer)

xlmr_model = XLMRClassifier("xlm-roberta-base", num_classes).to(device)
optimizer = optim.AdamW(xlmr_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

start = time.time()
best_f1 = 0.0
history = defaultdict(list)

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS} [XLM-Roberta]")
    xlmr_model, train_acc, train_loss = train_model(train_loader, xlmr_model, optimizer)
    val_metrics = eval_model(val_loader, xlmr_model, target_list)
    val_f1 = val_metrics['f1']
    val_loss = val_metrics['loss']
    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_f1'].append(val_f1)
    history['val_loss'].append(val_loss)

    scheduler.step()

    if val_f1 > best_f1:
        torch.save(xlmr_model.state_dict(), "xlmr_model.bin")
        best_f1 = val_f1
        print("✅ Best XLM-Roberta model saved!")

end = time.time()
print(f"⏱️ Total XLM-Roberta training & evaluation time: {end - start:.2f} seconds")

print("\nTesting Best XLM-Roberta Model...")
xlmr_model.load_state_dict(torch.load("xlmr_model.bin"))

start = time.time()  # Start test-time
metrics=eval_model(test_loader, xlmr_model,target_list)
print(metrics)
end = time.time()
print(f"🧪 XLM-Roberta test-set evaluation time: {end - start:.2f} seconds")


['anger' 'disgust' 'fear' 'joy' 'sadness' 'surprise']

Training: BanglaBERT


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

2025-05-13 05:15:23.982170: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747113324.181070      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747113324.242214      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

...


=== Final Test Metrics ===
Accuracy       : 0.6548
F1 Score       : 0.6541
Precision      : 0.6536
Recall         : 0.6548
AUC ROC Score  : 0.9104
Average Loss   : 1.0838

=== Classification Report ===
              precision    recall  f1-score   support

       anger       0.53      0.52      0.53      1518
     disgust       0.59      0.57      0.58      1310
        fear       0.70      0.70      0.70       756
         joy       0.82      0.83      0.82      1784
     sadness       0.62      0.61      0.62      1631
    surprise       0.66      0.70      0.68      1011

    accuracy                           0.65      8010
   macro avg       0.65      0.65      0.65      8010
weighted avg       0.65      0.65      0.65      8010


=== Per-Class Metrics ===
Class 0 (anger): Accuracy = 0.8213, AUC = 0.8602
Class 1 (disgust): Accuracy = 0.8639, AUC = 0.8921
Class 2 (fear): Accuracy = 0.9428, AUC = 0.9414
Class 3 (joy): Accuracy = 0.9211, AUC = 0.9635
Class 4 (sadness): Accuracy = 0.


=== Final Test Metrics ===
Accuracy       : 0.6582
F1 Score       : 0.6572
Precision      : 0.6564
Recall         : 0.6582
AUC ROC Score  : 0.9111
Average Loss   : 1.0828

=== Classification Report ===
              precision    recall  f1-score   support

       anger       0.54      0.51      0.53      1518
     disgust       0.58      0.58      0.58      1310
        fear       0.71      0.70      0.70       756
         joy       0.82      0.83      0.83      1784
     sadness       0.61      0.63      0.62      1631
    surprise       0.67      0.68      0.67      1011

    accuracy                           0.66      8010
   macro avg       0.66      0.66      0.66      8010
weighted avg       0.66      0.66      0.66      8010


=== Per-Class Metrics ===
Class 0 (anger): Accuracy = 0.8257, AUC = 0.8606
Class 1 (disgust): Accuracy = 0.8639, AUC = 0.8938
Class 2 (fear): Accuracy = 0.9439, AUC = 0.9428
Class 3 (joy): Accuracy = 0.9213, AUC = 0.9637
Class 4 (sadness): Accuracy = 0.


=== Final Test Metrics ===
Accuracy       : 0.6582
F1 Score       : 0.6572
Precision      : 0.6564
Recall         : 0.6582
AUC ROC Score  : 0.9111
Average Loss   : 1.0828

=== Classification Report ===
              precision    recall  f1-score   support

       anger       0.54      0.51      0.53      1518
     disgust       0.58      0.58      0.58      1310
        fear       0.71      0.70      0.70       756
         joy       0.82      0.83      0.83      1784
     sadness       0.61      0.63      0.62      1631
    surprise       0.67      0.68      0.67      1011

    accuracy                           0.66      8010
   macro avg       0.66      0.66      0.66      8010
weighted avg       0.66      0.66      0.66      8010


=== Per-Class Metrics ===
Class 0 (anger): Accuracy = 0.8257, AUC = 0.8606
Class 1 (disgust): Accuracy = 0.8639, AUC = 0.8938
Class 2 (fear): Accuracy = 0.9439, AUC = 0.9428
Class 3 (joy): Accuracy = 0.9213, AUC = 0.9637
Class 4 (sadness): Accuracy = 0.


=== Final Test Metrics ===
Accuracy       : 0.6552
F1 Score       : 0.6540
Precision      : 0.6533
Recall         : 0.6552
AUC ROC Score  : 0.9104
Average Loss   : 1.0946

=== Classification Report ===
              precision    recall  f1-score   support

       anger       0.53      0.52      0.53      1518
     disgust       0.60      0.55      0.57      1310
        fear       0.67      0.72      0.70       756
         joy       0.82      0.83      0.82      1784
     sadness       0.61      0.62      0.62      1631
    surprise       0.67      0.68      0.68      1011

    accuracy                           0.66      8010
   macro avg       0.65      0.66      0.65      8010
weighted avg       0.65      0.66      0.65      8010


=== Per-Class Metrics ===
Class 0 (anger): Accuracy = 0.8220, AUC = 0.8598
Class 1 (disgust): Accuracy = 0.8658, AUC = 0.8924
Class 2 (fear): Accuracy = 0.9408, AUC = 0.9431
Class 3 (joy): Accuracy = 0.9206, AUC = 0.9633
Class 4 (sadness): Accuracy = 0.


=== Final Test Metrics ===
Accuracy       : 0.6594
F1 Score       : 0.6575
Precision      : 0.6570
Recall         : 0.6594
AUC ROC Score  : 0.9076
Average Loss   : 1.0815

=== Classification Report ===
              precision    recall  f1-score   support

       anger       0.57      0.50      0.53      1518
     disgust       0.57      0.62      0.60      1310
        fear       0.71      0.72      0.71       757
         joy       0.79      0.83      0.81      1784
     sadness       0.62      0.61      0.62      1631
    surprise       0.69      0.68      0.68      1010

    accuracy                           0.66      8010
   macro avg       0.66      0.66      0.66      8010
weighted avg       0.66      0.66      0.66      8010


=== Per-Class Metrics ===
Class 0 (anger): Accuracy = 0.8328, AUC = 0.8610
Class 1 (disgust): Accuracy = 0.8628, AUC = 0.8928
Class 2 (fear): Accuracy = 0.9456, AUC = 0.9511
Class 3 (joy): Accuracy = 0.9119, AUC = 0.9573
Class 4 (sadness): Accuracy = 0.